# negative-back composite — cx6: negative_back — −grad_out under the canonical (grad_out, out, x) contract

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `negative-back`, `backward-fn-signature`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "negative-back"
DD_ATOM_IDS = ["negative-back", "backward-fn-signature"]
DD_SUBTOPICS = ["Backprop: negative_back", "Backprop: backward fn signature"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `negative_back` as a minimal signature-contract demo

1. **`backward-fn-signature`** — every back fn takes `(grad_out, out, *args)`
   and returns `dL/dargs[i]` with the SAME shape as the corresponding arg.
   The signature is uniform even when some fields go unused.
2. **`negative-back`** — `out = -x` has constant local derivative `-1`, so
   the chain rule collapses to `dL/dx = -grad_out`. Neither `out` nor `x` is
   read.

Composition: `negative_back` is the SIMPLEST possible back fn — useful
exactly because it forces you to honour the full signature even when half of
it is dead weight. The reverse-pass dispatcher must be able to call any back
fn the same way, so the signature contract is non-negotiable.


### Composite Exercise — negative_back — −grad_out under the canonical (grad_out, out, x) contract

**Atoms exercised together**: `negative-back`, `backward-fn-signature`

Implement `cx6_negative_back(grad_out, out, x)` for `out = -x`.

Requirements:
- Signature must be exactly `(grad_out, out, x)` — three positional args.
  The test calls it positionally AND introspects its signature.
- Return `-grad_out` (chain rule with constant local derivative `-1`).
- Must NOT mutate `grad_out` in place (the test re-uses the same tensor).
- Output shape must equal `x.shape` (which here equals `grad_out.shape`).
- Do not read `out` or `x` for the value — the local derivative is
  position-independent.


In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

def cx6_negative_back(grad_out, out, x):
    """dL/dx for out = -x. Honours (grad_out, out, x) signature."""
    raise NotImplementedError()


def _test_cx6():
    import inspect
    # --- signature contract: three positional params named (grad_out, out, x) ---
    sig = inspect.signature(cx6_negative_back)
    params = list(sig.parameters)
    assert params == ['grad_out', 'out', 'x'], (
        f'signature must be (grad_out, out, x), got {params}'
    )

    # --- scalar ---
    x = t.tensor([3.0])
    out = -x
    g = cx6_negative_back(t.tensor([1.0]), out, x)
    assert t.allclose(g, t.tensor([-1.0])), f'scalar: {g}'

    # --- vector + non-unit grad_out ---
    x = t.tensor([1.0, -2.0, 3.0])
    out = -x
    grad_out = t.tensor([5.0, 7.0, -2.0])
    g = cx6_negative_back(grad_out, out, x)
    assert g.shape == x.shape
    assert t.allclose(g, -grad_out), f'vector: {g}'

    # --- matrix shape ---
    rng = t.Generator().manual_seed(0)
    X = t.randn(3, 4, generator=rng)
    G = t.randn(3, 4, generator=rng)
    g = cx6_negative_back(G, -X, X)
    assert g.shape == (3, 4)
    assert t.allclose(g, -G)

    # --- non-mutation: grad_out must be unchanged by the call ---
    grad_in = t.tensor([1.0, 2.0, 3.0])
    grad_in_copy = grad_in.clone()
    _ = cx6_negative_back(grad_in, t.zeros(3), t.zeros(3))
    assert t.allclose(grad_in, grad_in_copy), 'must not mutate grad_out in place'

    # --- doesn't depend on out or x: scrambling them must not change the result ---
    go = t.tensor([4.0, -1.0, 8.0])
    real_x = t.tensor([1.0, 2.0, 3.0])
    g_real = cx6_negative_back(go, -real_x, real_x)
    g_fake = cx6_negative_back(go, t.tensor([99.0, 99.0, 99.0]), t.tensor([-7.0, 0.5, 1e6]))
    assert t.allclose(g_real, g_fake), (
        'negative_back must depend ONLY on grad_out — local derivative is constant -1'
    )

    # --- witness vs torch.autograd ---
    x_ref = t.tensor([1.5, -0.3, 2.7], requires_grad=True)
    (-x_ref).sum().backward()
    ours = cx6_negative_back(t.ones(3), -x_ref.detach(), x_ref.detach())
    assert t.allclose(ours, x_ref.grad, atol=1e-6)

    _dd_passed.add('cx6')

_test_cx6()

<details><summary>Show solution — cx6</summary>

```python
def cx6_negative_back(grad_out, out, x):
    # d/dx (-x) = -1; chain rule → -grad_out. `out` and `x` go unread.
    return -grad_out

```

Two atoms: the parameter list IS the signature contract
(`backward-fn-signature`); the body `-grad_out` IS `negative-back`. The
scramble test confirms the body ignores `out`/`x`, while the introspection
test confirms the signature is still the canonical three-positional form.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx6',
        'subtopics': ["Backprop: negative_back", "Backprop: backward fn signature"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()